## Ingest Via Notebooks into Unity Catalog

### Setup the Unity Catalog Structure

In [ ]:
%sql

CREATE SCHEMA IF NOT EXISTS EV_lab.bronze
  COMMENT 'Raw ingested data — unmodified, as received from field systems';

CREATE SCHEMA IF NOT EXISTS EV_lab.silver
  COMMENT 'Cleansed and enriched data — ready for analytics and reporting';
  
CREATE VOLUME IF NOT EXISTS EV_lab.bronze.raw_files
  COMMENT 'Landing zone for raw CSV files from SCADA systems and field sensors';

### Write the Data to Unity Catalog Volumes

In [ ]:
import requests

# Define the current catalog
catalog_name = "EV_lab"

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/bronze/raw_files"

# List of files to download
files = ["EV_charging_sessions.csv", "EV_station_events.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DP-750/refs/heads/main/Data_Ingestion_Unity_Catalog/Data/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

### Load the EV Charging Sessions Data to a Spark Dataframe

In [ ]:
df = spark.read.load(f'/Volumes/{catalog_name}/bronze/raw_files/EV_charging_sessions.csv', format='csv')
display(df)

### Defining Schema for the Dataframe

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

evSessionSchema = StructType([
    StructField("session_id", StringType()),
    StructField("station_id", StringType()),
    StructField("charger_id", StringType()),
    StructField("start_time", TimestampType()),
    StructField("end_time", TimestampType()),
    StructField("energy_kwh", FloatType()),
    StructField("charging_rate_kw", FloatType()),
    StructField("temperature_c", FloatType()),
    StructField("status", StringType())
])

df_sessions = spark.read.load(
    f'/Volumes/{catalog_name}/bronze/raw_files/EV_charging_sessions.csv',
    format='csv',
    schema=evSessionSchema,
    header=True
)

display(df_sessions)

### Write the Dataframe to a Managed Delta Table

In [ ]:
from delta.tables import *
from pyspark.sql.functions import *

# Storing the Bronze Layer as a Delta Table in the Data Catalog
df_sessions.write.mode("overwrite").saveAsTable('EV_lab.bronze.ev_charging_sessions')